# Notebook 03 — Pipelines & Evaluation (Kaggle T4)

**Required Inputs:**
1. Kaggle Input dataset: `raddar/chest-xrays-indiana-university`
2. Previous notebook output (with `reports_corpus.csv`, `qa_dataset.jsonl`, `colpali_index/`, `clip_index/`)

**Required Secret:** `HF_TOKEN` (for MedGemma access)

Runs 3 systems on 50 test studies (reduced for time):
- System A: ColPali + MedGemma (RAG)
- System B: CLIP + MedGemma (RAG)
- System C: MedGemma Direct

In [1]:
!pip install -q --upgrade peft transformers
!pip install -q accelerate bitsandbytes colpali-engine open-clip-torch faiss-cpu
!pip install -q bert-score rouge-score
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 86.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 10.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.2 MB/s eta 0:00:00:00:01


In [2]:
import os, sys, subprocess, importlib.util, glob, time
import pandas as pd
from PIL import Image

WORKING_DIR = '/kaggle/working'

# Edit each path to match your Kaggle Input mount points
CORPUS_PATH       = '/kaggle/input/datasets/mohammedtaha778/reports-corpus/reports_corpus.csv'
QA_PATH           = '/kaggle/input/datasets/mohammedtaha778/reports-corpus/qa_dataset.jsonl'
COLPALI_INDEX_DIR = '/kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/colpali_index'
CLIP_INDEX_DIR    = '/kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/clip_index'

# Verify all exist
for path, name in [(CORPUS_PATH, 'corpus'), (QA_PATH, 'qa_dataset'),
                    (COLPALI_INDEX_DIR, 'colpali_index'), (CLIP_INDEX_DIR, 'clip_index')]:
    exists = '✓' if os.path.exists(path) else '✗'
    print(f'  {exists} {name}: {path}')
    if not os.path.exists(path):
        print(f'    ⚠️  Path does not exist — update the variable above')


  ✓ corpus: /kaggle/input/datasets/mohammedtaha778/reports-corpus/reports_corpus.csv
  ✓ qa_dataset: /kaggle/input/datasets/mohammedtaha778/reports-corpus/qa_dataset.jsonl
  ✓ colpali_index: /kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/colpali_index
  ✓ clip_index: /kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/clip_index


In [4]:
# Get HF_TOKEN from Kaggle secrets
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
print('✓ HF_TOKEN loaded')

✓ HF_TOKEN loaded


In [8]:
import os, sys, subprocess, importlib.util, glob, time
import pandas as pd
from PIL import Image

REPO_PATH = '/kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/cxr-rag-system'

# Clone if needed
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mohamedtaha77/cxr-rag-system.git', REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

# CRITICAL: Add repo to sys.path so `from src...` imports work
sys.path.insert(0, REPO_PATH)

# Now use normal imports
from src.generation.medgemma_generator import MedGemmaGenerator
from src.retrieval.colpali_retriever import ColPaliRetriever
from src.retrieval.clip_retriever import CLIPRetriever
from src.evaluation.metrics import Evaluator

print('✓ Modules loaded')

fatal: detected dubious ownership in repository at '/kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/cxr-rag-system'
To add an exception for this directory, call:

	git config --global --add safe.directory /kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/cxr-rag-system


CalledProcessError: Command '['git', '-C', '/kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/cxr-rag-system', 'pull', '-q']' returned non-zero exit status 128.

In [6]:
# Load corpus + test split (limit to 50 for time)
corpus_df = pd.read_csv(CORPUS_PATH)
test_df = corpus_df[corpus_df['split'] == 'test'].head(50).reset_index(drop=True)
study_to_impression = dict(zip(corpus_df['study_id'], corpus_df['impression']))

print(f'Evaluating on {len(test_df)} test studies')

Evaluating on 50 test studies


In [7]:
# Load MedGemma (4-bit, ~3 GB VRAM)
import torch, gc

torch.cuda.empty_cache()
gc.collect()

print('Loading MedGemma (4-bit)...')
generator = MedGemmaGenerator(hf_token=HF_TOKEN, load_in_4bit=True)
print('✓ MedGemma loaded')

Loading MedGemma (4-bit)...


NameError: name 'MedGemmaGenerator' is not defined

In [ ]:
# Helper function
def run_report_pipeline(row, retriever, study_to_impression, k=3):
    image = Image.open(row['image_path']).convert('RGB')
    query = row.get('impression', 'chest x-ray findings')[:100]
    retrieved = retriever.search(query, k=k)
    context = [
        study_to_impression.get(
            os.path.splitext(os.path.basename(r.get('image_path', '')))[0].replace('.dcm', ''), ''
        )
        for r in retrieved if r.get('image_path')
    ]
    context = [c for c in context if c][:k]
    return generator.generate_report(image, context_reports=context or None)

## System A: ColPali + MedGemma

In [ ]:
from tqdm import tqdm

print('Loading ColPali index...')
colpali = ColPaliRetriever.from_index(COLPALI_INDEX_DIR)
print('✓ Loaded')

preds_A, refs_A = [], []
start = time.time()
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='System A'):
    pred = run_report_pipeline(row, colpali, study_to_impression)
    preds_A.append(pred)
    refs_A.append(row['impression'])

elapsed_A = time.time() - start
print(f'✓ System A done in {elapsed_A/60:.1f} min')

del colpali
gc.collect()
torch.cuda.empty_cache()

## System B: CLIP + MedGemma

In [ ]:
print('Loading CLIP index...')
clip = CLIPRetriever()
clip.load_index(CLIP_INDEX_DIR)
print('✓ Loaded')

# CLIP uses search_by_text, need wrapper
class CLIPWrapper:
    def __init__(self, clip):
        self.clip = clip
    def search(self, query, k=3):
        return self.clip.search_by_text(query, k=k)

clip_wrapped = CLIPWrapper(clip)

preds_B, refs_B = [], []
start = time.time()
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='System B'):
    pred = run_report_pipeline(row, clip_wrapped, study_to_impression)
    preds_B.append(pred)
    refs_B.append(row['impression'])

elapsed_B = time.time() - start
print(f'✓ System B done in {elapsed_B/60:.1f} min')

del clip, clip_wrapped
gc.collect()
torch.cuda.empty_cache()

## System C: MedGemma Direct

In [ ]:
preds_C, refs_C = [], []
start = time.time()
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='System C'):
    image = Image.open(row['image_path']).convert('RGB')
    pred = generator.generate_report(image, context_reports=None)
    preds_C.append(pred)
    refs_C.append(row['impression'])

elapsed_C = time.time() - start
print(f'✓ System C done in {elapsed_C/60:.1f} min')

## Compute Metrics

In [ ]:
evaluator = Evaluator()

results = {}
for label, preds, refs in [
    ('ColPali + MedGemma', preds_A, refs_A),
    ('CLIP + MedGemma', preds_B, refs_B),
    ('MedGemma Direct', preds_C, refs_C),
]:
    print(f'Computing metrics for {label}...')
    results[label] = evaluator.evaluate_report_generation(preds, refs)

results_df = pd.DataFrame(results).T
print('\n=== Results ===')
print(results_df.round(4))

# Save to /kaggle/working/
results_df.to_csv(os.path.join(WORKING_DIR, 'results.csv'))

# Save predictions
predictions_df = pd.DataFrame({
    'study_id': test_df['study_id'].tolist(),
    'reference': refs_A,
    'colpali_pred': preds_A,
    'clip_pred': preds_B,
    'direct_pred': preds_C,
})
predictions_df.to_csv(os.path.join(WORKING_DIR, 'predictions.csv'), index=False)

print(f'\n✓ Saved results.csv and predictions.csv to {WORKING_DIR}')

## QA Evaluation

In [ ]:
# Load QA test set (limit to 30 for time)
qa_df = pd.read_json(QA_PATH, lines=True)
qa_test = qa_df[qa_df['split'] == 'test'].head(30).reset_index(drop=True)
print(f'QA evaluation on {len(qa_test)} pairs')

# Reload ColPali for QA
colpali = ColPaliRetriever.from_index(COLPALI_INDEX_DIR)

qa_preds, qa_refs = [], []
start = time.time()
for _, row in tqdm(qa_test.iterrows(), total=len(qa_test), desc='QA'):
    image = Image.open(row['image_path']).convert('RGB')
    retrieved = colpali.search(row['question'], k=3)
    context = [
        study_to_impression.get(
            os.path.splitext(os.path.basename(r.get('image_path', '')))[0].replace('.dcm', ''), ''
        )
        for r in retrieved if r.get('image_path')
    ]
    context = [c for c in context if c][:3]
    pred = generator.answer_question(image, row['question'], context)
    qa_preds.append(pred)
    qa_refs.append(row['answer'])

elapsed_qa = time.time() - start
print(f'✓ QA done in {elapsed_qa/60:.1f} min')

qa_metrics = evaluator.evaluate_qa(qa_preds, qa_refs)
print(f'\nQA Metrics (ColPali + MedGemma): {qa_metrics}')

qa_results_df = pd.DataFrame({
    'question': qa_test['question'].tolist(),
    'reference': qa_refs,
    'prediction': qa_preds,
})
qa_results_df.to_csv(os.path.join(WORKING_DIR, 'qa_results.csv'), index=False)
print(f'\n✓ Saved qa_results.csv to {WORKING_DIR}')